**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

**Load Dataset**

In [ ]:
# Load preprocessed crime dataset
df = pd.read_csv("CrimeData_v2.csv")
df

,crime,location,date,sex,victim_ethnicity,victim_age,time,weather,latitude,longitude,holiday_name,is_holiday,land_use_type,gn_division,gn_pcode,gn_population,gn_distance_m,victim_ethnicity,status_report
0,burglary,mulgampola,2019-12-31,f,muslim,54,08:17:00,Cloudy,7.280544,80.616500,Non-Holiday,0,General Urban,Welata,LK2130170,21826,311.8,muslim,Valid
1,burglary,car park,2020-01-04,m,sinhala,42,02:00:00,Rainy,7.283445,80.619385,Non-Holiday,0,Commercial,Katukele West,LK2130105,8913,352.4,sinhala,Valid
2,theft,bus stand,2020-01-08,f,sinhala,20,21:01:00,Rainy,7.256425,80.590461,Eid al-Adha,1,Commercial,Penideniya,LK2139135,16411,518.6,sinhala,Valid
3,vehicle theft,aniwatte,2020-01-10,m,sinhala,29,12:10:00,Cloudy,7.290058,80.622438,Adhi Vap Full Moon Poya Day,1,General Urban,Aniwatta East,LK2130100,1107,381.5,sinhala,Valid
4,robbery,dutugamunu mawatha,2020-01-11,m,sinhala,59,02:39:00,Rainy,7.312344,80.645687,Non-Holiday,0,General Urban,Aruppala East,LK2130050,1293,322.6,sinhala,Valid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1603,drugs,walespark,2025-07-30,m,sinhala,65,09:19:00,Rainy,7.291772,80.636707,Non-Holiday,0,Commercial,Mahanuwara,LK2130120,11613,518.0,sinhala,Valid
1604,drugs,peradeniya road,2025-08-16,m,sinhala,59,04:38:00,Rainy,7.280930,80.619477,Non-Holiday,0,General Urban,Suduhumpala West,LK2130165,18808,353.7,sinhala,Valid
1605,robbery,provincial education department,2025-09-13,f,sinhala,50,07:25:00,Rainy,7.290341,80.633563,Non-Holiday,0,Commercial,Bogambara,LK2130145,2198,315.0,sinhala,Valid
1606,robbery,post office,2025-09-17,f,sinhala,61,20:12:00,Rainy,7.292186,80.633475,Non-Holiday,0,Commercial,Ihala Katukele,LK2130115,6925,335.7,sinhala,Valid


In [ ]:
# Quick inspection
print(df.shape)
print(df.columns)

(1608, 19)
Index(['crime', 'location', 'date', 'sex', 'victim_ethnicity', 'victim_age',
       'time', 'weather', 'latitude', 'longitude', 'holiday_name',
       'is_holiday', 'land_use_type', 'gn_division', 'gn_pcode',
       'gn_population', 'gn_distance_m', 'victim_ethnicity ', 'status_report'],
      dtype='object')


In [ ]:
# Check for columns with similar names
cols_with_spaces = [col for col in df.columns if col.endswith(' ')]
print("Columns with trailing space:", cols_with_spaces)

# Strip whitespace
df.columns = df.columns.str.strip()

# If there are actual duplicates after stripping
if df.columns.duplicated().any():
    print("Duplicate columns found after cleaning!")
    # Keep first occurrence, drop subsequent duplicates
    df = df.loc[:, ~df.columns.duplicated()]

Columns with trailing space: ['victim_ethnicity ']
Duplicate columns found after cleaning!


In [ ]:
df.columns

Index(['crime', 'location', 'date', 'sex', 'victim_ethnicity', 'victim_age',
       'time', 'weather', 'latitude', 'longitude', 'holiday_name',
       'is_holiday', 'land_use_type', 'gn_division', 'gn_pcode',
       'gn_population', 'gn_distance_m', 'status_report'],
      dtype='object')

**Select component relevant columns**

In [ ]:
required_cols = [
    'crime',
    'date',
    'time',
    'location',
    'latitude',
    'longitude',
    'land_use_type',
    'weather',
    'is_holiday',

]

df = df[required_cols].copy()


**Standardize text columns**

In [ ]:
df['crime'] = df['crime'].str.lower().str.strip()
df['location'] = df['location'].str.lower().str.strip()

**Ensure date and time types**

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.time

**Extract time features**

In [ ]:
df['hour'] = df['time'].apply(lambda x: x.hour)
df['day_of_week'] = df['date'].dt.dayofweek  # 0=Mon
df['month'] = df['date'].dt.month

**Create time window**

In [ ]:
def time_bucket(hour):
    if 0 <= hour < 6:
        return 'late_night'
    elif 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    else:
        return 'night'

In [ ]:
df['time_bucket'] = df['hour'].apply(time_bucket)

**Location categorization**

In [ ]:
def categorize_location(loc):
    if pd.isna(loc):
        return 'unknown'

    if any(x in loc for x in ['bus stand', 'train station', 'junction','watarauma', 'bridge']):
        return 'transport_hub'

    elif any(x in loc for x in ['hospital', 'clinic']):
        return 'medical'

    elif any(x in loc for x in ['road', 'street', 'lane', 'mawatha']):
        return 'road'

    elif any(x in loc for x in ['school', 'college', 'campus','primary']):
        return 'education'

    elif any(x in loc for x in ['market', 'bank', 'complex']):
        return 'commercial'

    elif any(x in loc for x in ['hotel', 'kfc', 'crown']):
        return 'hospitality'

    elif any(x in loc for x in ['stadium', 'ground']):
        return 'recreation'

    elif any(x in loc for x in ['police', 'prison', 'department','office']):
        return 'government'

    elif any(x in loc for x in ['temple', 'maligawa']):
        return 'religious'

    elif any(x in loc for x in ['cemetery', 'malshalawa']):
        return 'cemetery'

    elif 'car park' in loc:
        return 'parking'

    elif any(x in loc for x in ['town', 'city centre']):
        return 'urban_center'

    elif loc.replace(' ', '').isalpha():
        return 'residential'

    else:
        return 'other'

In [ ]:
df['location_category'] = df['location'].apply(categorize_location)

In [ ]:
df

,crime,date,time,location,latitude,longitude,land_use_type,weather,is_holiday,hour,day_of_week,month,time_bucket,location_category
0,burglary,2019-12-31,08:17:00,mulgampola,7.280544,80.616500,General Urban,Cloudy,0,8,1,12,morning,residential
1,burglary,2020-01-04,02:00:00,car park,7.283445,80.619385,Commercial,Rainy,0,2,5,1,late_night,parking
2,theft,2020-01-08,21:01:00,bus stand,7.256425,80.590461,Commercial,Rainy,1,21,2,1,night,transport_hub
3,vehicle theft,2020-01-10,12:10:00,aniwatte,7.290058,80.622438,General Urban,Cloudy,1,12,4,1,afternoon,residential
4,robbery,2020-01-11,02:39:00,dutugamunu mawatha,7.312344,80.645687,General Urban,Rainy,0,2,5,1,late_night,road
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1603,drugs,2025-07-30,09:19:00,walespark,7.291772,80.636707,Commercial,Rainy,0,9,2,7,morning,residential
1604,drugs,2025-08-16,04:38:00,peradeniya road,7.280930,80.619477,General Urban,Rainy,0,4,5,8,late_night,road
1605,robbery,2025-09-13,07:25:00,provincial education department,7.290341,80.633563,Commercial,Rainy,0,7,5,9,morning,government
1606,robbery,2025-09-17,20:12:00,post office,7.292186,80.633475,Commercial,Rainy,0,20,2,9,night,government


**Create day type feature**

In [ ]:
df["day_type"] = df["is_holiday"].apply(
    lambda x: "holiday" if x == 1 else "non_holiday"
)

**Remove rows without coordinates**

In [ ]:
df = df.dropna(subset=["latitude", "longitude"])

**Encode features for pattern**

In [ ]:
#one-hot encoding
crime_ohe = pd.get_dummies(df['crime'], prefix='crime')
time_ohe = pd.get_dummies(df['time_bucket'], prefix='time')
location_ohe = pd.get_dummies(df['location_category'], prefix='loc')
day_dummies = pd.get_dummies(df['day_type'], prefix='day')
weather_dummies = pd.get_dummies(df['weather'], prefix='weather')
land_use_dummies = pd.get_dummies(df['land_use_type'], prefix='land_use')

**Create transaction data set**

In [ ]:
transaction_df = pd.concat(
    [
        crime_ohe,
        time_ohe,
        location_ohe,
        day_dummies,
        weather_dummies,
        land_use_dummies
    ],
    axis=1
)
transaction_df = transaction_df.astype(int)

In [ ]:
# Drop the binary column, keep one-hot columns
df = df.drop(['is_holiday'], axis=1)
df = df.drop(['weather'], axis=1)
df = df.drop(['land_use_type'], axis=1)


**Final dataset**

In [ ]:
final_df = pd.concat(
    [
        df,
        transaction_df
    ],
    axis=1
)
final_df.head()


,crime,date,time,location,latitude,longitude,hour,day_of_week,month,time_bucket,...,loc_urban_center,day_holiday,day_non_holiday,weather_Cloudy,weather_Rainy,weather_Sunny,land_use_Commercial,land_use_General Urban,land_use_Recreational,land_use_Residential
0,burglary,2019-12-31,08:17:00,mulgampola,7.280544,80.616500,8,1,12,morning,...,0,0,1,1,0,0,0,1,0,0
1,burglary,2020-01-04,02:00:00,car park,7.283445,80.619385,2,5,1,late_night,...,0,0,1,0,1,0,1,0,0,0
2,theft,2020-01-08,21:01:00,bus stand,7.256425,80.590461,21,2,1,night,...,0,1,0,0,1,0,1,0,0,0
3,vehicle theft,2020-01-10,12:10:00,aniwatte,7.290058,80.622438,12,4,1,afternoon,...,0,1,0,1,0,0,0,1,0,0
4,robbery,2020-01-11,02:39:00,dutugamunu mawatha,7.312344,80.645687,2,5,1,late_night,...,0,0,1,0,1,0,0,1,0,0


In [ ]:
df['location_category'].value_counts()

,count
location_category,
residential,566
road,377
transport_hub,182
urban_center,131
commercial,83
government,51
religious,49
hospitality,49
medical,36


**Save to csv**

In [ ]:
final_df.to_csv(
    "pattern_recognition_preprocessed.csv",
    index=False
)

print("Pattern recognition preprocessing completed and saved.")

Pattern recognition preprocessing completed and saved.
